# Spacing_vs_Powers

Y Axis: Laser Power

X Axis: Horizontal line spacing

Feed Rate: (Determined from previous experiments)

Purpose: Inspect half toning / darkness of each square.

In [21]:
import gcode
import numpy as np

In [22]:
def init(**kwargs) -> gcode.GCode:
    prog = gcode.GCode(**kwargs)
    prog.G21()
    prog.G90()
    # Zero on the K30 cross hairs.
    prog.G92(X=0, Y=0)
    # Set motion velocity.
    prog.G1(F=1000)
    prog.G0(F=1000)

    prog.M3(S=0)
    # Cross hairs offset.
    prog.G1(X=16.5, Y=1)
    prog.G92(X=0, Y=0)

    # Offset for axes labels
    prog.G1(X=10, Y=10)
    prog.G92(X=0, Y=0)

    prog.M5()
    return prog

Draw the Axis

In [35]:
Xf = 100
Yf = 40


def add_axes():
    # X Axis
    prog = gcode.GCode()
    prog += gcode.Line(
        points=gcode.hline(X0=0, Xf=Xf, Y=0), power=100, feed=1000, dynamic_power=False
    )
    prog += gcode.Line(
        points=gcode.vline(Y0=0, Yf=Yf, X=0), power=100, feed=1000, dynamic_power=False
    )
    # Offset to start drawing rectangles.
    prog.G1(X=5, Y=5)
    # Re-zero for the experiment
    prog.G92(X=0, Y=0)

    return prog

In [45]:
powers = np.arange(10, 100 + 1, 10)

spacings = [0.5, 0.75, 1, 1.5, 2]

In [51]:
prog = init()
prog += add_axes()
speed = 500
square_size = 10

Y = 0
X = 0
for spacing_idx, spacing in enumerate(spacings):
    X = 0
    for power_idx, power in enumerate(powers):
        for y_idx, y in enumerate(np.arange(0, 10 + spacing, spacing)):
            if np.mod(y_idx, 2) == 0:
                prog += gcode.Line(
                    points=gcode.hline(X0=X + square_size, Xf=X, Y=Y + y),
                    power=power,
                    feed=speed,
                    dynamic_power=False,
                )
            else:
                prog += gcode.Line(
                    points=gcode.hline(X0=X, Xf=X + square_size, Y=Y + y),
                    power=power,
                    feed=speed,
                    dynamic_power=False,
                )
        X += square_size
    Y += square_size

prog.M5()
prog.G1(X=-10, Y=-10, F=2000)
prog.G92(X=0, Y=0)

prog.save("/tmp/Spacing_vs_Power.gcode")

In [50]:
Y

50